In [2]:
import pandas as pd
import mysql.connector
from pathlib import Path

# 1) 파일 경로
file_path = "C:/skn29/PYTHON/project/data/자동차보험_손해상황_전체_2023_2025.xlsx"

# 2) 엑셀 읽기
df = pd.read_excel(file_path)

print("원본 행 수:", len(df))
print("원본 컬럼:", df.columns.tolist())

# 3) 필요한 컬럼만 선택
df = df[
    [
        'isuCmpyOfrYm',
        'isuItmsNm',
        'mogClsfNm',
        'kncrNm',
        'losAmt',
        'injPtlCnt',
        'dthTotlCnt'
    ]
].copy()

# 4) 컬럼명 변경
df = df.rename(columns={
    'isuCmpyOfrYm': 'stat_year_month',
    'isuItmsNm': 'insurance_product_name',
    'mogClsfNm': 'coverage_name',
    'kncrNm': 'vehicle_class',
    'losAmt': 'loss_amount',
    'injPtlCnt': 'injury_partial_count',
    'dthTotlCnt': 'death_total_loss_count'
})

# 5) 문자열 컬럼 공백 제거
str_cols = [
    'stat_year_month',
    'insurance_product_name',
    'coverage_name',
    'vehicle_class'
]

for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# 6) 숫자형 변환
df['stat_year_month'] = pd.to_numeric(df['stat_year_month'], errors='coerce')
df['loss_amount'] = pd.to_numeric(df['loss_amount'], errors='coerce')
df['injury_partial_count'] = pd.to_numeric(df['injury_partial_count'], errors='coerce')
df['death_total_loss_count'] = pd.to_numeric(df['death_total_loss_count'], errors='coerce')

# 7) 결측 제거
df = df.dropna(subset=[
    'stat_year_month',
    'insurance_product_name',
    'coverage_name',
    'vehicle_class',
    'loss_amount',
    'injury_partial_count',
    'death_total_loss_count'
])

# 8) 타입 변환
df['stat_year_month'] = df['stat_year_month'].astype(int).astype(str)
df['loss_amount'] = df['loss_amount'].astype(int)
df['injury_partial_count'] = df['injury_partial_count'].astype(int)
df['death_total_loss_count'] = df['death_total_loss_count'].astype(int)

# 9) 값 필터링
allowed_products = ['개인용', '업무용', '영업용']
allowed_coverages = ['대인배상1', '대인배상2', '자기신체사고', '대물배상', '자기차량손해']
allowed_vehicle_classes = ['소형', '중형', '대형', '다인승']

df = df[df['insurance_product_name'].isin(allowed_products)].copy()
df = df[df['coverage_name'].isin(allowed_coverages)].copy()
df = df[df['vehicle_class'].isin(allowed_vehicle_classes)].copy()

print("필터 후 행 수:", len(df))
print(df['vehicle_class'].value_counts())

# 10) MySQL 연결
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='!didwjdgus16',
    database='car_insurance_final',
    charset='utf8mb4'
)

cursor = conn.cursor(dictionary=True)

# 11) 코드 테이블 매핑
cursor.execute("""
SELECT insurance_product_id, insurance_product_name
FROM insurance_product
""")
product_map = {
    row['insurance_product_name']: row['insurance_product_id']
    for row in cursor.fetchall()
}

cursor.execute("""
SELECT coverage_id, coverage_name
FROM coverage_master
""")
coverage_map = {
    row['coverage_name']: row['coverage_id']
    for row in cursor.fetchall()
}

# 12) ID 매핑
df['insurance_product_id'] = df['insurance_product_name'].map(product_map)
df['coverage_id'] = df['coverage_name'].map(coverage_map)

# 13) 매핑 실패 제거
df = df.dropna(subset=['insurance_product_id', 'coverage_id']).copy()
df['insurance_product_id'] = df['insurance_product_id'].astype(int)
df['coverage_id'] = df['coverage_id'].astype(int)

# 14) 최종 컬럼만 선택
df = df[
    [
        'stat_year_month',
        'insurance_product_id',
        'coverage_id',
        'vehicle_class',
        'loss_amount',
        'injury_partial_count',
        'death_total_loss_count'
    ]
].copy()

# 15) 유니크 키 기준 중복 제거
df = df.drop_duplicates(subset=[
    'stat_year_month',
    'insurance_product_id',
    'coverage_id',
    'vehicle_class'
])

print("최종 적재 대상 행 수:", len(df))
print(df.head())

# 16) INSERT SQL
insert_sql = """
INSERT INTO insurance_loss_stat (
    stat_year_month,
    insurance_product_id,
    coverage_id,
    vehicle_class,
    loss_amount,
    injury_partial_count,
    death_total_loss_count
) VALUES (%s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    loss_amount = VALUES(loss_amount),
    injury_partial_count = VALUES(injury_partial_count),
    death_total_loss_count = VALUES(death_total_loss_count),
    updated_at = CURRENT_TIMESTAMP
"""

# 17) executemany용 데이터 생성
data_to_insert = [
    (
        row.stat_year_month,
        row.insurance_product_id,
        row.coverage_id,
        row.vehicle_class,
        row.loss_amount,
        row.injury_partial_count,
        row.death_total_loss_count
    )
    for row in df.itertuples(index=False)
]

# 18) INSERT 실행
cursor.close()
cursor = conn.cursor()
cursor.executemany(insert_sql, data_to_insert)
conn.commit()

print(f"{cursor.rowcount}건 처리 완료")

# 19) 결과 확인
cursor.execute("SELECT COUNT(*) FROM insurance_loss_stat")
print("insurance_loss_stat 전체 건수:", cursor.fetchone()[0])

cursor.execute("""
SELECT
    stat_year_month,
    insurance_product_id,
    coverage_id,
    vehicle_class,
    loss_amount,
    injury_partial_count,
    death_total_loss_count
FROM insurance_loss_stat
ORDER BY stat_year_month DESC
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

원본 행 수: 2450
원본 컬럼: ['isuCmpyOfrYm', 'isuItmsNm', 'mogClsfNm', 'kncrNm', 'losAmt', 'injPtlCnt', 'dthTotlCnt']
필터 후 행 수: 700
vehicle_class
대형     175
중형     175
소형     175
다인승    175
Name: count, dtype: int64
최종 적재 대상 행 수: 700
   stat_year_month  insurance_product_id  coverage_id vehicle_class  \
0           202511                     1            5            대형   
21          202511                     1            5            중형   
22          202511                     1            5            소형   
50          202511                     1            4           다인승   
51          202511                     1            4            대형   

     loss_amount  injury_partial_count  death_total_loss_count  
0   723953066000                299903                   11783  
21  906052825000                370432                   19203  
22  619649295000                407682                   22288  
50  407670357000                207447                    2710  
51  800378564000      